# Trees and ensembles: partitions, instability, and aggregation

**Question.** Do axis-aligned chemistry rules separate cultivars, and what is gained by combining many trees?

A decision tree is readable as nested rules but can change sharply under small data perturbations. Random forests reduce variance through randomized aggregation; boosting builds a sequence that emphasizes prior errors. We compare these behaviors through one shared package boundary.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.tree import plot_tree

from dsm_project.config import load_config
from dsm_project.data import load_project_dataset
from dsm_project.evaluation import comparison_table
from dsm_project.experiments import fit_supervised_candidate

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)
task = dataset.tasks["classification"]

table = comparison_table(
    dataset,
    config,
    task_name="classification",
    model_names=("decision_tree", "random_forest", "gradient_boosting"),
)
display(table)

## Recursive partitioning

At a node, a classifier considers rules such as $x_j\le t$ and chooses a split that improves class purity according to a declared criterion. Repeating this process partitions feature space into rectangles and assigns each leaf a prediction.

Deep trees have low training bias but high variance. Pruning, depth, leaf-size and feature restrictions regularize the partition. A random forest averages many decorrelated trees fitted to bootstrap and feature subsets. Gradient boosting adds weak trees sequentially to reduce an objective. These mechanisms are not interchangeable even when all use trees.

In [ ]:
plot_table = table.set_index("model")["value"].sort_values()
plot_table.plot.barh(color=["#1b9e77", "#7570b3", "#d95f02"])
plt.xlabel("balanced accuracy on instructional holdout")
plt.ylabel("")
plt.title("A family comparison on one shared split")
plt.xlim(0, 1.05)
plt.tight_layout();

In [ ]:
tree_result = fit_supervised_candidate(
    dataset, config, task_name="classification", model_name="decision_tree"
)
forest_result = fit_supervised_candidate(
    dataset, config, task_name="classification", model_name="random_forest"
)

tree = tree_result.estimator.named_steps["decisiontreeclassifier"]
plt.figure(figsize=(14, 6))
plot_tree(
    tree,
    feature_names=list(tree_result.feature_names),
    class_names=[str(value) for value in tree.classes_],
    filled=True,
    max_depth=2,
    rounded=True,
)
plt.title("Top of the fitted reference tree")
plt.tight_layout();

forest = forest_result.estimator.named_steps["randomforestclassifier"]
importance = pd.Series(forest.feature_importances_, index=forest_result.feature_names)
display(importance.sort_values(ascending=False).head(8).to_frame("impurity importance"))

## Interpretability needs a contract

The displayed rules describe this fitted tree, not a universal law of wine chemistry. Impurity importance can favor continuous or high-cardinality features and does not show causal effect or even direction. Correlated features can divide credit unpredictably. Use held-out permutation or domain-specific perturbation carefully, and explain the population and decision to which an interpretation applies.

## Conclusion, limits, and student implementation

One split cannot establish an ensemble advantage. A professional comparison uses repeated or nested evidence, retains the single-tree and simple baselines, measures uncertainty and serving cost, and tests stability across perturbations.

Student code should implement and test a split criterion, deterministic tie rules, stopping conditions, leaf predictions, and traversal before moving to ensembles. For bagging or forests, test bootstrap and feature-sampling behavior plus aggregation. For boosting, derive the objective-specific update. Keep visualization in the notebook and reusable tree behavior in the package.